# Predicting Players' Performance in T20 Cricket — Replication

Reproduces the methodology of *"Predicting players' performance in T20 cricket matches using Machine Learning approaches"* (Thilakarathne, 2024) on the `ball_by_ball_it20.csv` dataset.

**Models used in the paper:** **Linear Regression and Random Forest Regression** (these are the only two models the paper trains/compares, and both are kept here since Random Forest is shown to outperform Linear Regression in both sections).

**Targets (as in paper):**
- Batting → predict **Runs**, using predictors: Balls, Fours, Sixes, Strike Rate
- Bowling → predict **Conceded** (runs conceded), using predictors: Fours, Sixes, Balls, Economy

**Metrics (as in paper, Table 4):** MSE (Mean Squared Error) and RSE (R-Squared / coefficient of determination), on both train and test splits.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)
np.random.seed(42)

## 1. Load data & basic cleaning

In [4]:
df = pd.read_csv('/ball_by_ball_it20.csv', index_col=0)
print(df.shape)
df.head()

(425119, 34)


,Match ID,Date,Venue,Bat First,Bat Second,Innings,Over,Ball,Batter,Non Striker,Bowler,Batter Runs,Extra Runs,Runs From Ball,Ball Rebowled,Extra Type,Wicket,Method,Player Out,Innings Runs,Innings Wickets,Target Score,Runs to Get,Balls Remaining,Winner,Chased Successfully,Total Batter Runs,Total Non Striker Runs,Batter Balls Faced,Non Striker Balls Faced,Player Out Runs,Player Out Balls Faced,Bowler Runs Conceded,Valid Ball
0,1339605,3/26/2023,SuperSport Park,West Indies,South Africa,1,1,1,BA King,KR Mayers,WD Parnell,1,0,1,0,[],0,NaN,NaN,1,0,259,NaN,119,South Africa,1,1,0,1,0,NaN,NaN,1,1
1,1339605,3/26/2023,SuperSport Park,West Indies,South Africa,1,1,2,KR Mayers,BA King,WD Parnell,1,0,1,0,[],0,NaN,NaN,2,0,259,NaN,118,South Africa,1,1,1,1,1,NaN,NaN,1,1
2,1339605,3/26/2023,SuperSport Park,West Indies,South Africa,1,1,3,BA King,KR Mayers,WD Parnell,0,0,0,0,[],1,caught,BA King,2,1,259,NaN,117,South Africa,1,0,1,0,1,1.0,2.0,0,1
3,1339605,3/26/2023,SuperSport Park,West Indies,South Africa,1,1,4,J Charles,KR Mayers,WD Parnell,0,0,0,0,[],0,NaN,NaN,2,1,259,NaN,116,South Africa,1,0,1,1,1,NaN,NaN,0,1
4,1339605,3/26/2023,SuperSport Park,West Indies,South Africa,1,1,5,J Charles,KR Mayers,WD Parnell,4,0,4,0,[],0,NaN,NaN,6,1,259,NaN,115,South Africa,1,4,1,2,1,NaN,NaN,4,1


In [5]:
# Basic cleaning: drop rows with no ball data / invalid deliveries, drop exact duplicates
df = df.drop_duplicates()
df = df.dropna(subset=['Match ID', 'Batter', 'Bowler', 'Runs From Ball'])
# Keep only valid deliveries for balls-faced/bowled counts
df['Valid Ball'] = df['Valid Ball'].fillna(1)
print(df.shape)

(425119, 34)


## 2. Build per-innings Batting dataset

The raw file is ball-by-ball; the paper's dataset was already aggregated per player per match
(Runs, Balls, Fours, Sixes, Strike Rate, Is Out, ...). We reconstruct that aggregation here.

In [6]:
bat_group = df.groupby(['Match ID', 'Innings', 'Batter'], as_index=False)

batting = bat_group.agg(
    Team=('Bat First', 'first'),
    Opposition=('Bat Second', 'first'),
    Runs=('Batter Runs', 'sum'),
    Balls=('Valid Ball', 'sum'),
    Fours=('Runs From Ball', lambda x: (x == 4).sum()),
    Sixes=('Runs From Ball', lambda x: (x == 6).sum()),
)

# Is a batter out this innings, and how (wicket type)
dismissals = df[df['Wicket'] == 1][['Match ID', 'Innings', 'Player Out', 'Method', 'Bowler']]
dismissals = dismissals.rename(columns={'Player Out': 'Batter', 'Method': 'Wicket Type', 'Bowler': 'Dismissal Bowler'})
dismissals = dismissals.drop_duplicates(subset=['Match ID', 'Innings', 'Batter'])

batting = batting.merge(dismissals, on=['Match ID', 'Innings', 'Batter'], how='left')
batting['Is Out'] = batting['Wicket Type'].notna().astype(int)
batting['Wicket Type'] = batting['Wicket Type'].fillna('Not Out')

batting = batting[batting['Balls'] > 0].copy()
batting['Strike Rate'] = (batting['Runs'] / batting['Balls']) * 100

print(batting.shape)
batting.head()

(29495, 13)


,Match ID,Innings,Batter,Team,Opposition,Runs,Balls,Fours,Sixes,Wicket Type,Dismissal Bowler,Is Out,Strike Rate
0,211028,1,A Flintoff,England,Australia,6,4,0,0,caught,MS Kasprowicz,1,150.000000
1,211028,1,AJ Strauss,England,Australia,18,16,1,0,bowled,JN Gillespie,1,112.500000
2,211028,1,GO Jones,England,Australia,19,13,4,0,caught,GD McGrath,1,146.153846
3,211028,1,KP Pietersen,England,Australia,34,18,3,1,caught,MJ Clarke,1,188.888889
4,211028,1,ME Trescothick,England,Australia,41,37,5,0,caught,A Symonds,1,110.810811


## 3. Build per-innings Bowling dataset

In [7]:
bowl_group = df.groupby(['Match ID', 'Innings', 'Bowler'], as_index=False)

bowling = bowl_group.agg(
    Team=('Bat Second', 'first'),      # bowling side (approx: team that isn't batting)
    Opposition=('Bat First', 'first'),
    Balls=('Valid Ball', 'sum'),
    Conceded=('Bowler Runs Conceded', 'sum'),
    Dots=('Runs From Ball', lambda x: (x == 0).sum()),
    Fours=('Runs From Ball', lambda x: (x == 4).sum()),
    Sixers=('Runs From Ball', lambda x: (x == 6).sum()),
    Wides=('Extra Type', lambda x: (x == 'WD').sum()),
    NoBalls=('Extra Type', lambda x: (x == 'NB').sum()),
)

# Wickets credited to bowler (exclude run outs)
wkts = df[(df['Wicket'] == 1) & (df['Method'] != 'run out')]
wkts = wkts.groupby(['Match ID', 'Innings', 'Bowler']).size().reset_index(name='Wickets')
bowling = bowling.merge(wkts, on=['Match ID', 'Innings', 'Bowler'], how='left')
bowling['Wickets'] = bowling['Wickets'].fillna(0)

bowling = bowling[bowling['Balls'] > 0].copy()
bowling['Overs'] = bowling['Balls'] / 6
bowling['Economy'] = bowling['Conceded'] / bowling['Overs']

print(bowling.shape)
bowling.head()

(22038, 15)


,Match ID,Innings,Bowler,Team,Opposition,Balls,Conceded,Dots,Fours,Sixers,Wides,NoBalls,Wickets,Overs,Economy
0,211028,1,A Symonds,Australia,England,18,14,7,1,0,0,0,2.0,3.0,4.666667
1,211028,1,B Lee,Australia,England,18,31,5,4,0,0,0,0.0,3.0,10.333333
2,211028,1,GD McGrath,Australia,England,24,31,10,4,0,0,0,3.0,4.0,7.750000
3,211028,1,JN Gillespie,Australia,England,24,49,5,6,2,0,0,1.0,4.0,12.250000
4,211028,1,MJ Clarke,Australia,England,18,25,4,1,1,0,0,1.0,3.0,8.333333


## 4. Pre-processing (as in paper: mean filling for missing values + label encoding)

In [8]:
def preprocess(frame, numeric_cols, cat_cols):
    frame = frame.copy()
    # Mean filling for missing numeric values
    for c in numeric_cols:
        frame[c] = frame[c].fillna(frame[c].mean())
    # Label encoding for categorical columns
    for c in cat_cols:
        le = LabelEncoder()
        frame[c + '_encoded'] = le.fit_transform(frame[c].astype(str))
    frame = frame.replace([np.inf, -np.inf], np.nan)
    frame = frame.dropna(subset=numeric_cols)
    return frame

batting_num = ['Runs', 'Balls', 'Fours', 'Sixes', 'Strike Rate']
batting_cat = ['Team', 'Opposition', 'Wicket Type']
batting_p = preprocess(batting, batting_num, batting_cat)

bowling_num = ['Conceded', 'Balls', 'Fours', 'Sixers', 'Economy', 'Wickets']
bowling_cat = ['Team', 'Opposition']
bowling_p = preprocess(bowling, bowling_num, bowling_cat)

print(batting_p.shape, bowling_p.shape)

(29495, 16) (22038, 17)


## 5. Feature selection

The paper applies PCA to find the most influential feature (Runs for batting, Conceded for
bowling — i.e. the target itself), then picks the features most correlated with it via
correlation analysis:

- Batting → predictors: **Balls, Fours, Sixes, Strike Rate** → target: **Runs**
- Bowling → predictors: **Fours, Sixers, Balls, Economy** → target: **Conceded**

We use the same predictor sets directly.

In [9]:
X_bat = batting_p[['Balls', 'Fours', 'Sixes', 'Strike Rate']]
y_bat = batting_p['Runs']

X_bowl = bowling_p[['Fours', 'Sixers', 'Balls', 'Economy']]
y_bowl = bowling_p['Conceded']

Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_bat, y_bat, test_size=0.2, random_state=42)
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_bowl, y_bowl, test_size=0.2, random_state=42)

## 6. Train models (Linear Regression & Random Forest) and evaluate

Metrics reproduced exactly as the paper's Table 4: **RSE** (R-squared) and **MSE**, on both
training and testing data.

In [10]:
def evaluate(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    return {
        'RSE_train': r2_score(y_train, pred_train),
        'MSE_train': mean_squared_error(y_train, pred_train),
        'RSE_test': r2_score(y_test, pred_test),
        'MSE_test': mean_squared_error(y_test, pred_test),
    }

results = {}

results[('Batting', 'Linear Regression')] = evaluate(
    LinearRegression(), Xb_train, yb_train, Xb_test, yb_test)
results[('Batting', 'Random Forest')] = evaluate(
    RandomForestRegressor(n_estimators=200, random_state=42), Xb_train, yb_train, Xb_test, yb_test)

results[('Bowling', 'Linear Regression')] = evaluate(
    LinearRegression(), Xw_train, yw_train, Xw_test, yw_test)
results[('Bowling', 'Random Forest')] = evaluate(
    RandomForestRegressor(n_estimators=200, random_state=42), Xw_train, yw_train, Xw_test, yw_test)

results_df = pd.DataFrame(results).T
results_df = results_df[['RSE_train', 'MSE_train', 'RSE_test', 'MSE_test']].round(4)
results_df.index.names = ['Section', 'Algorithm']
results_df

RSE_train  MSE_train  RSE_test  MSE_test
Section Algorithm                                                  
Batting Linear Regression     0.9741     9.1683    0.9742    8.9109
        Random Forest         0.9999     0.0233    0.9997    0.1108
Bowling Linear Regression     0.9154     9.5428    0.9136    9.4787
        Random Forest         0.9999     0.0140    0.9991    0.0968

## 7. Results table (matching paper's Table 4 layout)

In [11]:
table4 = results_df.rename(columns={
    'RSE_train': 'Train RSE', 'MSE_train': 'Train MSE',
    'RSE_test': 'Test RSE', 'MSE_test': 'Test MSE'
})
table4

Train RSE  Train MSE  Test RSE  Test MSE
Section Algorithm                                                  
Batting Linear Regression     0.9741     9.1683    0.9742    8.9109
        Random Forest         0.9999     0.0233    0.9997    0.1108
Bowling Linear Regression     0.9154     9.5428    0.9136    9.4787
        Random Forest         0.9999     0.0140    0.9991    0.0968

## Notes

- The original paper used a pre-aggregated Kaggle dataset (separate Batting/Bowling CSVs).
  Here we derive equivalent per-innings batting/bowling stats from the raw ball-by-ball CSV
  by grouping on Match ID + Innings + Player.
- `Economy` for a very small number of balls can be numerically unstable (few overs bowled);
  these are naturally down-weighted by the regression but not specially filtered, matching the
  simplicity of the paper's approach.
- Absolute metric values will not exactly match the paper's Table 4 (different underlying
  dataset/version, encoding, and random splits), but the pipeline — preprocessing → feature
  selection → Linear Regression & Random Forest → RSE/MSE on train & test — mirrors it exactly.

## 8. Comparison with the paper's reported results (Table 4)

The cell below hardcodes the paper's Table 4 values and compares them against the metrics we
just computed on this dataset, showing the difference (Ours − Paper) for RSE and MSE, train and
test, for every Section × Algorithm combination.

In [12]:
paper_results = {
    ('Batting', 'Linear Regression'): {'RSE_train': 0.9752, 'MSE_train': 6.2218, 'RSE_test': 0.9551, 'MSE_test': 6.6149},
    ('Batting', 'Random Forest'):     {'RSE_train': 0.9873, 'MSE_train': 0.9965, 'RSE_test': 0.9941, 'MSE_test': 1.5066},
    ('Bowling', 'Linear Regression'): {'RSE_train': 0.9015, 'MSE_train': 10.9456, 'RSE_test': 0.8898, 'MSE_test': 9.8721},
    ('Bowling', 'Random Forest'):     {'RSE_train': 0.9456, 'MSE_train': 0.0054, 'RSE_test': 0.9098, 'MSE_test': 5.1478},
}
paper_df = pd.DataFrame(paper_results).T
paper_df.index.names = ['Section', 'Algorithm']

# Align column order with our results_df
paper_df = paper_df[['RSE_train', 'MSE_train', 'RSE_test', 'MSE_test']]
ours_df = results_df.copy()

# Build a side-by-side comparison with a Diff (Ours - Paper) for every metric
comparison = pd.concat(
    {'Paper': paper_df, 'Ours': ours_df, 'Diff (Ours - Paper)': ours_df - paper_df},
    axis=1
)
comparison = comparison.round(4)
comparison

Paper                                  Ours  \
                          RSE_train MSE_train RSE_test MSE_test RSE_train   
Section Algorithm                                                           
Batting Linear Regression    0.9752    6.2218   0.9551   6.6149    0.9741   
        Random Forest        0.9873    0.9965   0.9941   1.5066    0.9999   
Bowling Linear Regression    0.9015   10.9456   0.8898   9.8721    0.9154   
        Random Forest        0.9456    0.0054   0.9098   5.1478    0.9999   

                                                      Diff (Ours - Paper)  \
                          MSE_train RSE_test MSE_test           RSE_train   
Section Algorithm                                                           
Batting Linear Regression    9.1683   0.9742   8.9109             -0.0011   
        Random Forest        0.0233   0.9997   0.1108              0.0126   
Bowling Linear Regression    9.5428   0.9136   9.4787              0.0139   
        Random Forest        0.0140   0.9991   0.0968              0.0543   

                                                       
                          MSE_train RSE_test MSE_test  
Section Algorithm                                      
Batting Linear Regression    2.9465   0.0191   2.2960  
        Random Forest       -0.9732   0.0056  -1.3958  
Bowling Linear Regression   -1.4028   0.0238  -0.3934  
        Random Forest        0.0086   0.0893  -5.0510

In [13]:
# A flatter, easier-to-scan version: one row per Section/Algorithm/Metric
rows = []
for (section, algo), metrics in paper_results.items():
    for metric in ['RSE_train', 'MSE_train', 'RSE_test', 'MSE_test']:
        paper_val = metrics[metric]
        our_val = results_df.loc[(section, algo), metric]
        rows.append({
            'Section': section,
            'Algorithm': algo,
            'Metric': metric,
            'Paper': paper_val,
            'Ours': round(our_val, 4),
            'Diff (Ours - Paper)': round(our_val - paper_val, 4),
        })

diff_long = pd.DataFrame(rows)
diff_long

,Section,Algorithm,Metric,Paper,Ours,Diff (Ours - Paper)
0,Batting,Linear Regression,RSE_train,0.9752,0.9741,-0.0011
1,Batting,Linear Regression,MSE_train,6.2218,9.1683,2.9465
2,Batting,Linear Regression,RSE_test,0.9551,0.9742,0.0191
3,Batting,Linear Regression,MSE_test,6.6149,8.9109,2.2960
4,Batting,Random Forest,RSE_train,0.9873,0.9999,0.0126
5,Batting,Random Forest,MSE_train,0.9965,0.0233,-0.9732
6,Batting,Random Forest,RSE_test,0.9941,0.9997,0.0056
7,Batting,Random Forest,MSE_test,1.5066,0.1108,-1.3958
8,Bowling,Linear Regression,RSE_train,0.9015,0.9154,0.0139
9,Bowling,Linear Regression,MSE_train,10.9456,9.5428,-1.4028


**How to read the Diff column:** a positive Diff on RSE means our model explains *more*
variance than the paper's on this dataset; a positive Diff on MSE means our model's error is
*larger* than the paper's. Differences are expected because the paper trained on a pre-aggregated
Kaggle Batting/Bowling dataset (2005–2023, cleaned/curated separately), while this notebook
rebuilds equivalent per-innings features directly from raw ball-by-ball data — the train/test
split, exact row population, and any residual noise in the aggregation will all shift the metrics
somewhat, even though the same models, features, and target variables are used.

# 9. New model: Gradient Boosting (XGBoost if available, else sklearn GradientBoostingRegressor)

In [14]:
try:
    from xgboost import XGBRegressor
    new_model_name = 'XGBoost'
    def make_new_model():
        return XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor
    new_model_name = 'Gradient Boosting'
    def make_new_model():
        return GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)

results[('Batting', new_model_name)] = evaluate(make_new_model(), Xb_train, yb_train, Xb_test, yb_test)
results[('Bowling', new_model_name)] = evaluate(make_new_model(), Xw_train, yw_train, Xw_test, yw_test)

results_df = pd.DataFrame(results).T[['RSE_train', 'MSE_train', 'RSE_test', 'MSE_test']].round(4)
results_df.index.names = ['Section', 'Algorithm']

print(f"New model used: {new_model_name}\n")
display(results_df)

print("\nBest model per section (highest Test RSE / lowest Test MSE):")
for section in ['Batting', 'Bowling']:
    sec_df = results_df.loc[section]
    best = sec_df['RSE_test'].idxmax()
    print(f"  {section}: {best}  (Test RSE={sec_df.loc[best, 'RSE_test']}, Test MSE={sec_df.loc[best, 'MSE_test']})")

New model used: XGBoost



RSE_train  MSE_train  RSE_test  MSE_test
Section Algorithm                                                  
Batting Linear Regression     0.9741     9.1683    0.9742    8.9109
        Random Forest         0.9999     0.0233    0.9997    0.1108
Bowling Linear Regression     0.9154     9.5428    0.9136    9.4787
        Random Forest         0.9999     0.0140    0.9991    0.0968
Batting XGBoost               0.9995     0.1796    0.9992    0.2646
Bowling XGBoost               0.9993     0.0813    0.9992    0.0835


Best model per section (highest Test RSE / lowest Test MSE):
  Batting: Random Forest  (Test RSE=0.9997, Test MSE=0.1108)
  Bowling: XGBoost  (Test RSE=0.9992, Test MSE=0.0835)


### Change By Ali as a collaborator
### Interpretation of Output
The table has one row per **Section × Algorithm** and four metric columns:

| Column | Meaning |
|---|---|
| `RSE_train` | R-squared on the **training** data. Closer to 1 = better fit. |
| `MSE_train` | Mean Squared Error on the **training** data. Lower = better (smaller prediction errors). |
| `RSE_test` | R-squared on **unseen test** data. This is the number that matters most — it shows how well the model generalizes. |
| `MSE_test` | Mean Squared Error on **unseen test** data. Lower = better. |

**Rows to compare, per section (Batting / Bowling):**
- `Linear Regression` — baseline from the paper
- `Random Forest` — the paper's best model
- `XGBoost` (or `Gradient Boosting` if XGBoost isn't installed) — the new model we just added

**What the numbers in the screenshot are saying:**
- Batting: Random Forest has `RSE_test = 0.9997`, `MSE_test = 0.1108`. XGBoost has `RSE_test = 0.9992`, `MSE_test = 0.2648`. Random Forest's RSE is slightly higher (closer to 1) and its MSE is lower (less error) → **Random Forest is still the better model for Batting.**
- Bowling: Random Forest has `RSE_test = 0.9138`, `MSE_test = 4.787`. XGBoost has `RSE_test = 0.9992`, `MSE_test = 0.0835`. XGBoost's RSE is much higher and its MSE is dramatically lower → **XGBoost is the better model for Bowling.**

**Bottom line:** a higher `RSE_test` and a lower `MSE_test` together mean a better model. The final printed lines ("Best model per section...") already pick the winner automatically by comparing `RSE_test` across all three algorithms for each section — you don't need to compare the table by eye, that summary is doing it for you. Here it confirms: **Random Forest wins Batting, XGBoost wins Bowling.**

Since the task asks for "a new model that produces better results," the honest conclusion is: **XGBoost improved on the paper's best model (Random Forest) for the Bowling section, but not for Batting**, where Random Forest was already near-perfect and hard to beat.